In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

# Externals
from src.utils import *
import config
import warnings
warnings.filterwarnings('ignore')

In [18]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')
orig = read_csv_file('data/orig/f1_strategy_dataset_v4.csv')
orig = orig.dropna(axis=0)

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)
orig = reduce_mem_usage(orig)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)
Shape of orig data: (101305, 16)


In [19]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

orig_features = orig.drop([config.TARGET], axis=1)
orig_features = orig_features.dropna(axis=0)

### Blending

In [20]:
# HistGBM
oof_proba_hist = read_csv_file(r'artifacts\oof\HISTGBM_v2_oof_proba.csv')
oof_proba_hist = oof_proba_hist.iloc[:, 1:]

test_proba_hist = read_csv_file(r'artifacts\test_proba\HISTGBM_v2_test_proba.csv')
test_proba_hist = test_proba_hist.iloc[:, 1:]

# XGBM
oof_proba_xgbm = read_csv_file(r'artifacts\oof\XGB_v13_seed42_oof_proba.csv')
oof_proba_xgbm = oof_proba_xgbm.iloc[:, 1:]

test_proba_xgbm = read_csv_file(r'artifacts\test_proba\XGB_v13_seed42_test_proba.csv')
test_proba_xgbm = test_proba_xgbm.iloc[:, 1:]

# LGBM
oof_proba_lgbm = read_csv_file(r'artifacts\oof\LGBM_v4_seed42_oof_proba.csv')
oof_proba_lgbm = oof_proba_lgbm.iloc[:, 1:]

test_proba_lgbm = read_csv_file(r'artifacts\test_proba\LGBM_v4_seed42_test_proba.csv')
test_proba_lgbm = test_proba_lgbm.iloc[:, 1:]

# ResNet
oof_proba_resnet = read_csv_file(r'artifacts\oof\RESNET_v1_seed42_oof_proba.csv')
oof_proba_resnet = oof_proba_resnet.iloc[:, 1:]

test_proba_resnet = read_csv_file(r'artifacts\test_proba\RESNET_v1_seed42_test_proba.csv')
test_proba_resnet = test_proba_resnet.iloc[:, 1:]

# REALMLP
oof_proba_realmlp = read_csv_file(r'artifacts\oof\REALMLP_v1_seed42_oof_proba.csv')
oof_proba_realmlp = oof_proba_realmlp.iloc[:, 1:]

test_proba_realmlp = read_csv_file(r'artifacts\test_proba\REALMLP_v1_seed42_test_proba.csv')
test_proba_realmlp = test_proba_realmlp.iloc[:, 1:]

# CatBoost
oof_proba_catboost = read_csv_file(r'artifacts\oof\CatGBM_v0_oof_proba.csv')
oof_proba_catboost = oof_proba_catboost.iloc[:, 1:]

test_proba_catboost = read_csv_file(r'artifacts\test_proba\CatGBM_v0_test_proba.csv')
test_proba_catboost = test_proba_catboost.iloc[:, 1:]

In [21]:
# OOF
oof_hist = oof_proba_hist.values
oof_xgbm = oof_proba_xgbm.values
oof_lgbm = oof_proba_lgbm.values
oof_resnet = oof_proba_resnet.values
oof_realmlp = oof_proba_realmlp.values
oof_catboost = oof_proba_catboost.values

# TEST
test_hist = test_proba_hist.values
test_xgbm = test_proba_xgbm.values
test_lgbm = test_proba_lgbm.values
test_resnet = test_proba_resnet.values
test_realmlp = test_proba_realmlp.values
test_catboost = test_proba_catboost.values

In [22]:
score_hist = roc_auc_score(target, oof_hist)
score_xgbm = roc_auc_score(target, oof_xgbm)
score_lgbm = roc_auc_score(target, oof_lgbm)
score_resnet = roc_auc_score(target, oof_resnet)
score_realmlp = roc_auc_score(target, oof_realmlp)
score_catboost = roc_auc_score(target, oof_catboost)

print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")
print(f"CATBOOST alone: {score_catboost:.6f}")

HISTGBM alone: 0.945228
XGBM alone: 0.950772
LGBM alone: 0.948235
RESNET alone: 0.942910
REALMLP alone: 0.954034
CATBOOST alone: 0.952650


### Weighted Blend

In [8]:
def objective(trial):

    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    w_resnet = trial.suggest_float("w_resnet", 0.0, 1.0)
    w_realmlp = trial.suggest_float("w_realmlp", 0.0, 1.0)
    w_catboost = trial.suggest_float("w_catboost", 0.0, 1.0)

    total = (
        w_hist +
        w_xgbm +
        w_lgbm +
        w_resnet +
        w_realmlp +
        w_catboost
    )

    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total
    w_resnet /= total
    w_realmlp /= total
    w_catboost /= total

    preds = (
        w_hist * oof_hist +
        w_xgbm * oof_xgbm +
        w_lgbm * oof_lgbm +
        w_resnet * oof_resnet +
        w_realmlp * oof_realmlp +
        w_catboost * oof_catboost
    )

    return roc_auc_score(target, preds.ravel())


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=config.SEED)
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True
)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"] +
    best["w_resnet"] +
    best["w_realmlp"] +
    best["w_catboost"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total
w_resnet = best["w_resnet"] / total
w_realmlp = best["w_realmlp"] / total
w_catboost = best["w_catboost"] / total

# -------- PRINT --------
best_single = max(
    score_hist,
    score_xgbm,
    score_lgbm,
    score_resnet,
    score_realmlp,
    score_catboost
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")
print(f"CATBOOST alone: {score_catboost:.6f}")

print(f"\nBest blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")
print(f"CATBOOST weight: {w_catboost:.4f} ({w_catboost:.1%})")

  0%|          | 0/5000 [00:00<?, ?it/s]


HISTGBM alone: 0.945228
XGBM alone: 0.950772
LGBM alone: 0.948235
RESNET alone: 0.942910
REALMLP alone: 0.954034
CATBOOST alone: 0.952650

Best blend: 0.954577
Gain: +0.000543
HIST weight: 0.0001 (0.0%)
XGBM weight: 0.1205 (12.0%)
LGBM weight: 0.0291 (2.9%)
RESNET weight: 0.0001 (0.0%)
REALMLP weight: 0.6432 (64.3%)
CATBOOST weight: 0.2072 (20.7%)


In [9]:
blend_oof = (
    w_hist * oof_hist +
    w_xgbm * oof_xgbm +
    w_lgbm * oof_lgbm +
    w_resnet * oof_resnet +
    w_realmlp * oof_realmlp +
    w_catboost * oof_catboost
)

blend_test = (
    w_hist * test_hist +
    w_xgbm * test_xgbm +
    w_lgbm * test_lgbm +
    w_resnet * test_resnet +
    w_realmlp * test_realmlp +
    w_catboost * test_catboost
)

# -------- SAVE BLENDED FILES --------
blend_oof_df = pd.DataFrame(
    blend_oof.ravel(),
    columns=['PitNextLap']
)

blend_oof_df.insert(0, "id", train_ids)

save_csv_file(
    blend_oof_df,
    os.path.join(
        config.OOF_PROBA_DIR,
        'weightblend_histV2_xgbV13_lgbV4_resnetV1_realmlpV1_catgbmV0_oof_proba[0.954577].csv'
    )
)

blend_test_df = pd.DataFrame(
    blend_test.ravel(),
    columns=['PitNextLap']
)

blend_test_df.insert(0, "id", test_ids)

save_csv_file(
    blend_test_df,
    os.path.join(
        config.TEST_PROBA_DIR,
        'weightblend_histV2_xgbV13_lgbV4_resnetV1_realmlpV1_catgbmV0_test_proba[0.954577].csv'
    )
)

In [15]:
# Sanity check
sanity_oof = read_csv_file(r'artifacts\oof\weightblend_histV2_xgbV13_lgbV4_resnetV1_realmlpV1_catgbmV0_oof_proba[0.954577].csv')
print(roc_auc_score(target, sanity_oof.iloc[:, 1]))

0.9545769894117928


### Weighted Ranked Blend

In [23]:
def objective(trial):
    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    w_resnet = trial.suggest_float("w_resnet", 0.0, 1.0)
    w_realmlp = trial.suggest_float("w_realmlp", 0.0, 1.0)
    w_catboost = trial.suggest_float("w_catboost", 0.0, 1.0)

    total = (
        w_hist +
        w_xgbm +
        w_lgbm +
        w_resnet +
        w_realmlp +
        w_catboost
    )

    # Normalize weights [0, 1]
    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total
    w_resnet /= total
    w_realmlp /= total
    w_catboost /= total

    preds = (
        w_hist * rankdata(oof_hist.ravel()) +
        w_xgbm * rankdata(oof_xgbm.ravel()) +
        w_lgbm * rankdata(oof_lgbm.ravel()) +
        w_resnet * rankdata(oof_resnet.ravel()) +
        w_realmlp * rankdata(oof_realmlp.ravel()) +
        w_catboost * rankdata(oof_catboost.ravel())
    )

    return roc_auc_score(target, preds)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True
)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"] +
    best["w_resnet"] +
    best["w_realmlp"] +
    best["w_catboost"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total
w_resnet = best["w_resnet"] / total
w_realmlp = best["w_realmlp"] / total
w_catboost = best["w_catboost"] / total

# -------- FINAL RANKED BLEND --------
blend_oof = (
    w_hist * rankdata(oof_hist.ravel()) +
    w_xgbm * rankdata(oof_xgbm.ravel()) +
    w_lgbm * rankdata(oof_lgbm.ravel()) +
    w_resnet * rankdata(oof_resnet.ravel()) +
    w_realmlp * rankdata(oof_realmlp.ravel()) +
    w_catboost * rankdata(oof_catboost.ravel())
)

blend_test = (
    w_hist * rankdata(test_hist.ravel()) +
    w_xgbm * rankdata(test_xgbm.ravel()) +
    w_lgbm * rankdata(test_lgbm.ravel()) +
    w_resnet * rankdata(test_resnet.ravel()) +
    w_realmlp * rankdata(test_realmlp.ravel()) +
    w_catboost * rankdata(test_catboost.ravel())
)

# -------- SCORES --------
best_single = max(
    score_hist,
    score_xgbm,
    score_lgbm,
    score_resnet,
    score_realmlp,
    score_catboost
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")
print(f"CATBOOST alone: {score_catboost:.6f}")

print(f"\nBest Ranked Blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")
print(f"CATBOOST weight: {w_catboost:.4f} ({w_catboost:.1%})")

  0%|          | 0/5000 [00:00<?, ?it/s]


HISTGBM alone: 0.945228
XGBM alone: 0.950772
LGBM alone: 0.948235
RESNET alone: 0.942910
REALMLP alone: 0.954034
CATBOOST alone: 0.952650

Best Ranked Blend: 0.954542
Gain: +0.000508
HIST weight: 0.0000 (0.0%)
XGBM weight: 0.0949 (9.5%)
LGBM weight: 0.0291 (2.9%)
RESNET weight: 0.0005 (0.0%)
REALMLP weight: 0.6312 (63.1%)
CATBOOST weight: 0.2443 (24.4%)


### Simple ranked average

In [39]:
# OOF ranked average
blend_oof = (
    rankdata(oof_hist.ravel()) +
    rankdata(oof_xgbm.ravel()) +
    rankdata(oof_lgbm.ravel()) +
    rankdata(oof_resnet.ravel()) +
    rankdata(oof_realmlp.ravel())
) / 5

# TEST ranked average
blend_test = (
    rankdata(test_hist.ravel()) +
    rankdata(test_xgbm.ravel()) +
    rankdata(test_lgbm.ravel()) +
    rankdata(test_resnet.ravel()) +
    rankdata(test_realmlp.ravel())
) / 5

# Score
score = roc_auc_score(target, blend_oof)

print(f"Ranked Blend AUC: {score:.6f}")

Ranked Blend AUC: 0.951187
